### HOW TO SETUP YOUR OWN RAG

Walkthrough

    Collecting data.
    
    Reading data.
    
    Chunking data.
    
    flattening data.
    
    load encoder.
    
    encode our data.
    
    load and setup chromadb for our encoded data.
    
    add our encoded data to chromadb.
    
    connect our local or online LLM to db.
    
    Test the model.

# Collecting data.

for my test data that will be provided i scarped some lastest news for Counter Strike , League of Legends and Valorant. each article stored separatly in text file.
we open every text file and store it in dictionary.[File_name][Article text]

In [19]:
import os
import re
def scan_openfile(path):
    all_files=[]
    for filename in os.listdir(path):
        if filename.endswith(".txt"):
            with open(os.path.join(path, filename), "r", encoding="utf-8") as f:
                lines = f.readlines()[7:]          # Skip first 7 lines as they are just metadata from our scarpping
                content = "".join(lines).lstrip()
                
                all_files.append({
                    "file_name":filename.removesuffix(".txt"), #remove .text suffice for file name
                    "content":content
                })
    return all_files


# Chunking function
## why do we chunk our texts?
Some documents are very long and may exceed the LLM's context window (the maximum number of tokens it can process at once). Chunking helps avoid this problem by splitting documents into smaller, manageable pieces.

Chunking also provides more accurate retrieval because we can retrieve only the relevant chunks instead of sending or retrieving the entire document.

We will also introduce overlap between chunks. This means that consecutive chunks will share some text, preserving context between them and preventing important information from being lost at chunk boundaries.

In [20]:
def chunking_function(text, chunk_size=200, overlap=30): 
    #We will use a simple regex based chunking, with chunk size at 200 token with 30 token overlap 
    #The number of tokens for chunk and overlap may change with different documents and texts ,it a trial and error procedure
    tokens = re.findall(r"\w+|[^\w\s]", text) #this allows to grab any word and non characters for our token
    chunks = []
    start = 0
    while start < len(tokens): #chunking method, we grab 230 token for our first chunk(200 size + 30 overlap)
        end = start + chunk_size

        chunk = tokens[start:end]
        chunks.append(" ".join(chunk)) #this make all the retrieved chunks in one single list 
        
        #the next start would be at 200 token (230-30(overlap) and the end would be at 400)
        start += chunk_size - overlap

    return chunks

In [21]:
path=["league_of_legends","cs2","valorant"] #our paths or what ever path for your text
all_texts=[]
for each in path:
    all_text=scan_openfile(each)
    #as our predefined function it scans and add a dictionary with items being structured like this[article_name][content]
    all_texts.extend(all_text)

import re
all_chunk=[]
for i in all_texts:
    chunk=chunking_function(i["content"]) #pass content tag to item and return chunk vector
    all_chunk.append(
        {
        "file_name":i["file_name"], #add file name + chunks
        "chunks":chunk
        }
    )

In [23]:
print("you can see the final result is a list with 60 articles as we have in total with our dictionary keys being filename and chunks")
print(type(all_chunk))     
print(len(all_chunk))
print(type(all_chunk[0]))       
print(all_chunk[0].keys() if isinstance(all_chunk[0], dict) else all_chunk[0])


you can see the final result is a list with 60 articles as we have in total with our dictionary keys being filename and chunks
<class 'list'>
60
<class 'dict'>
dict_keys(['file_name', 'chunks'])


In [36]:
all_chunk[0]["chunks"][4] #we can verify that our chunking and storing strategy is working correctly

"rest of the year means we ' ll still see something new from K / DA . I just hope Riot doesn ' t forget Pentakill ; they pioneered this , after all . Valorant ' s got some metal going on this year with If The Sun Burns Out , so let ' s hope League does too ."

# Flattening
To pass this to our chromadb our list should be all flattened to do this
we need to genarate unique index for each chunk

In [37]:

documents = []
metadatas = []
ids = []

for doc in all_chunk:
    for i, chunk in enumerate(doc["chunks"]): #enumarate function let us retrieve correct index and actual text

        documents.append(chunk)

        metadatas.append({
            "file_name": doc["file_name"],
            "chunk_index": i
        })

        ids.append(f'{doc["file_name"]}_{i}')
#The result of this is how we will store our flattened news articles in our chroma DB in the following STEP

# ChromaDB is an open-source vector database built  to store, index, and query vector embeddings for artificial intelligence and large language model (LLM) applications

In [41]:
import chromadb
#we create a localinstance for our chromadb
client = chromadb.PersistentClient(
    path="./chroma_db"
)
#create a database or a collection
collection = client.get_or_create_collection(
    name="test_rag"
)

## Encoding , our retrieved documents must be encoded we will use a simple encoder enough for it


In [40]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    documents,
    convert_to_numpy=True
)

/home/vaak/ai/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 715.08it/s]


In [42]:
#we add our data for our collection and it will be ready to be used on a encoder searching or last step which is RAG
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

In [52]:
#Test example query
results = collection.query(
    query_embeddings=[model.encode("what happend to aleksib ").tolist()],
    n_results=3
)

for doc, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("File:", metadata["file_name"])
    print("Chunk:", metadata["chunk_index"])
    print("Distance:", distance)
    print("Content:", doc)
    print("-" * 40)

File: 03-cs2-pro-aleksib-facing-suspended-prison-sentence-over-alleged-tax
Chunk: 1
Distance: 0.9566633701324463
Content: earnings were not declared but maintains that he did not know the money was subject to taxation . The allegations center on money Aleksib received through Counter - Strike sticker sales , which allow players and teams competing at Majors to earn a share of revenue from in - game purchases . By 2020 , Aleksib had competed at two Majors with ENCE , including the IEM Katowice 2019 Major , where the Finnish roster reached the grand final . According to court documents reported by Iltalehti , authorities allege that Aleksib deliberately omitted € 53 , 000 in sticker revenue from his 2020 tax declaration . However , the player reportedly disputes that the omission was intentional , arguing that he was unaware the sticker earnings needed to be reported as taxable income . Prosecutors are seeking a four - month suspended prison sentence , meaning Aleksib would not immediate

## when searching for aleksib you can see the retieved chunks are all from it news article which implies our chunking and storing was correct and it gives better results 
the next step is to connect our LLM to chromaDB and instruct it to follow the data from chromaDB to answer us.

Im using a local LLM that is compatible with openai library for easier use

In [57]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",
    api_key="no-key"
)
response = client.chat.completions.create( #testing that our local is connected and working
    model="local-model",
    messages=[
        {"role": "user", "content": "what happend to aleksib ."}
    ],
    temperature=0.7,
    max_tokens=1000
)

print(response.choices[0].message.content)

The name "Aleksib" is not widely recognized as a prominent historical figure, artist, or public figure in mainstream contexts. However, there are a few possibilities:

1. **Historical Figure**:  
   - **Aleksandar Aleksibović** (1445–1476) was a Serbian nobleman and literary patron from the 15th century. He was killed in 1476 during the Battle of the Sava River, and his death is recorded in historical accounts. His literary works and cultural contributions are noted.

2. **Modern Context**:  
   - "Aleksib" could refer to a contemporary individual, such as a musician, artist, or public figure. However, without additional context (e.g., field, location, or time period), it's difficult to specify their activities.

3. **Fictional or Anecdotal References**:  
   - The name might appear in literature, films, or other media, but there is no widely known narrative about Aleksib in mainstream sources.

If you meant a specific person or context, please provide more details!


## from our local model (qwen 1.7B) it doesn't have any information regarding alexib 

# we will now connect our local LLM to chromaDB and produce our own costume chatbot

In [63]:
def rag_query(question):

    # 1. Embed the question
    query_embedding = model.encode(question).tolist()

    # 2. Retrieve relevant chunks (As previously used when we just conducted searches)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    # 3. Extract retrieved text
    context = "\n\n".join(results["documents"][0])

    # 4. Send context + question to Qwen
    response = client.chat.completions.create(
        model="local-model",
        messages=[ #Our query instruction to make sure our LLM doesn't derail or hallucinate 
            {
                "role": "system",
                "content": """You are a helpful assistant.
                Answer the user's question using only the
                provided context. If the answer is not
                available, say you don't know."""
            },
            {
                "role": "user",
                "content": f"""
                Context:
                {context}

                Question:
                {question}
                """
            }
        ],
        temperature=0.7, #you can expirement but a higher temp means more strictness while a lower one means model can get creative
        max_tokens=1000
    )

    return response.choices[0].message.content

In [67]:
answer = [rag_query("what happened to s1mple ? answer in just one sentence and point which document from rag"),
          rag_query("cs2 crosshair"),
          rag_query("what happened to bin ? answer in just one sentence and point which document from rag"),]
for each in answer:
    print("\n \n")
    print(each)


 

The context does not mention S1mple's specific events, so the answer is that the information is not available in the provided document.

 

In Counter-Strike 2 (CS2), the best crosshairs for pro players often involve static crosshairs that remain fixed when firing or moving, as dynamic ones can be distracting. To find the best crosshairs, use the "Filter Player" and "Filter Team" functions to locate pro players, then copy their crosshair codes. These codes can be imported into CS2 by pasting them into the game settings, which will instantly apply the crosshair. The crosshair size may vary depending on the resolution (e.g., 16:9, 16:10, 4:3), and some pros use 4:3 stretched resolution for larger crosshairs. Additionally, pro players often change their crosshairs regularly, so the list is updated as new codes emerge. For a personalized setup, you can "copy my code" before importing a new one to retain your previous crosshair. If you observe another player's crosshair during a match, 

#### you can see in the end our local rag correctly retrieve informations from correct documents
### while this is just a simple beginner tutorial but it covers what necessary for you to learn on how to setup your own chatbot, 
### you can get deep into this by expirementing with different tokenizers ( ai tokenizers or so) expirement on different models such as extracting information from pdfs and such, use other LLMs, combing rag with automated workflow or agents and many stuff
### hope this was helpful!